# Notebook 3 -- Modeling

**Purpose**: Build and evaluate machine learning models to predict which side of a retardant drop is protected (pt=1) vs. fire origin (pt=0)





In [1]:
# Importing Necessary Modules
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [2]:
# Loading extracted features and merging them
# Reloading from saved CSV files to prevent stale varible issues
df_dob  = pd.read_csv("training_features_dob.csv")
df_fuel = pd.read_csv("training_features_fuel.csv")
df_split = pd.read_csv("training_features_split.csv")


# Mergeing fuel features into DOB features on shared identifiers
df = df_dob.merge(
    df_fuel,
    on=["event_id", "buff_id", "drop_id"],
    how="left"
)
df = df.merge(
    df_split,
    on=["event_id", "buff_id", "drop_id"],
    how="left"
)

print(f"Total rows: {len(df)}")
print(f"Rows with split features: {df['split_burned_pixels'].notna().sum()}")

print(f"PT distribution:\n{df['pt'].value_counts()}")

Total rows: 21547
Rows with split features: 20582
PT distribution:
pt
0    10865
1    10682
Name: count, dtype: int64


In [3]:
# Preparing a Base DataFrame
# Filter to clean drop pairs (exactly 2 rows per drop_id)
# Drops with 1, 3, 4, or 6 rows (~2.4%) are excluded — ambiguous pairs
group_sizes = df.groupby("drop_id").size()
clean_drop_ids = group_sizes[group_sizes == 2].index
df_model = df[df["drop_id"].isin(clean_drop_ids)].copy()
print(f"Clean pairs: {len(df_model)} rows ({df_model['drop_id'].nunique()} drops)")

# Fix NaN mean_dob/median_dob/split_mean_dob — polygons where nothing burned.
# These rows are valid training examples (burned_pixels=0 already captures this).
# Filling with 0 keeps them rather than silently dropping rows.
df_model["mean_dob"]       = df_model["mean_dob"].fillna(0)
df_model["median_dob"]     = df_model["median_dob"].fillna(0)
df_model["split_mean_dob"] = df_model["split_mean_dob"].fillna(0)

# Building Paired-Difference Feature
# For each drop pair, compute how much each side differs from the pair's average.
# Positive diff = more than sibling, negative diff = less than sibling.
# This is the key insight: relative comparison between sides matters more than
# absolute values, because it's self-normalizing across drops of very different sizes.

# DOB paired-difference features (local buffer)
for col in ["unburned_pixels", "early_pixels", "burned_pixels"]:
    df_model[f"{col.replace('_pixels', '')}_pair_diff"] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())

# Fuel paired-difference features
for col in ["grass_pixels", "shrub_pixels", "timber_pixels", "nonburnable_pixels"]:
    df_model[f"{col.replace('_pixels', '')}_pair_diff"] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())

# Split paired-difference features (whole-raster comparison — strongest signal)
for col in ["split_unburned_pixels", "split_early_pixels", "split_burned_pixels"]:
    new_col = col.replace("split_", "split_").replace("_pixels", "") + "_pair_diff"
    df_model[new_col] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())


# New feature addition trying to fix the error. 25% of the ~10% error was due to disagreement between local pixels and split pixels information. we are trying to see the disagreement and train accordingly. 

df_model["signal_agreement"] = (
    df_model["burned_pair_diff"] * df_model["split_burned_pair_diff"]
)
df_model["local_signal_strength"] = df_model["burned_pair_diff"].abs()
df_model["split_signal_strength"] = df_model["split_burned_pair_diff"].abs()
df_model["split_dominance"] = (
    df_model["split_signal_strength"] /
    (df_model["local_signal_strength"] + df_model["split_signal_strength"] + 1e-9)
)

print(f"DataFrame shape after feature engineering: {df_model.shape}")

Clean pairs: 20474 rows (10237 drops)
DataFrame shape after feature engineering: (20474, 43)


In [ ]:
# Defining Feature Sets

raw_features = [
    "mean_dob", "median_dob", "early_pixels", "late_pixels",
    "unburned_pixels", "burned_pixels", "total_pixels"
]

diff_features = [
    "unburned_pair_diff", "early_pair_diff", "burned_pair_diff"
]

dob_features = raw_features + diff_features

fuel_features = [
    "dominant_fuel", "fuel_diversity",
    "grass_pixels", "shrub_pixels", "timber_pixels",
    "slash_pixels", "nonburnable_pixels",
    "grass_pair_diff", "shrub_pair_diff",
    "timber_pair_diff", "nonburnable_pair_diff"
]

# Whole-raster split features — the strongest predictor discovered.
# Same statistics as DOB features, but computed across the ENTIRE fire raster
# split by the extended drop line, rather than just the small buffer polygon.
split_features = [
    "split_mean_dob", "split_early_pixels", "split_late_pixels",
    "split_unburned_pixels", "split_burned_pixels", "split_total_pixels",
    "split_burned_pair_diff", "split_unburned_pair_diff", "split_early_pair_diff"
]



all_features = dob_features + fuel_features + split_features


# New fix after error analysis

agreement_features = [
    "signal_agreement",
    "local_signal_strength",
    "split_signal_strength",
    "split_dominance"
]

new_features = dob_features + split_features + agreement_features + fuel_features


In [ ]:
#Final Clean DataFrame
# Drop rows with NaN in any feature we'll use.
# Note: dominant_fuel NaN = polygon outside fuel data coverage (very rare)
df_model_clean = df_model.dropna(subset=new_features + ["pt"])
print(f"Rows for training: {len(df_model_clean)}")
print(f"Unique fires: {df_model_clean['event_id'].nunique()}")

# Define y, groups, gkf ONCE 
# These are defined here and never redefined below.
# All model comparisons use the same split for fair evaluation.
y      = df_model_clean["pt"]      # target: 1=protected, 0=origin
groups = df_model_clean["event_id"] # split by FIRE to prevent leakage
gkf    = GroupKFold(n_splits=5)

Rows for training: 20472
Unique fires: 420


In [6]:
# ---- Model Training and Evaluation Function ----
# Trains Random Forest across all 5 GroupKFold folds.
# Accuracy reported as mean ± std across folds for a stable,
# reliable estimate not dependent on one particular split.

def run_model_cv(feature_list, label):
    X = df_model_clean[feature_list]
    accs = []        # accuracy per fold
    importances = [] # feature importances per fold

    for fold, (train_idx, test_idx) in enumerate(gkf.split(df_model_clean, y, groups)):
        # 200 trees for stable predictions, random_state=42 for reproducibility
        model = RandomForestClassifier(n_estimators=200, random_state=42)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])     # train on training rows only
        preds = model.predict(X.iloc[test_idx])             # predict on unseen test rows
        accs.append(accuracy_score(y.iloc[test_idx], preds))
        importances.append(model.feature_importances_)

    mean_acc = np.mean(accs)
    std_acc  = np.std(accs)
    # average feature importances across all 5 folds (axis=0 = average across folds)
    mean_imp = np.mean(importances, axis=0)

    print(f"\n{label}")
    print(f"Accuracy: {mean_acc:.3f} ± {std_acc:.3f}")
    print(f"Per-fold: {[round(a, 3) for a in accs]}")
    importance_dict = dict(zip(feature_list, mean_imp))
    print("Top features:", dict(sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)[:5]))
    print("-" * 50)
    return mean_acc, std_acc, model

# Approach 1 - Simple Rule Baseline

Before any ML, we will test the simplest possible rule: **whichever side has more unburned pixels is the protected side.**

Physically motivated - pilots drop retardant ahead of the fire, so the protected side should have more unburned area. Evaluated only on clean drop pairs (exactly 2 sides per drop). 


In [7]:
# Within each drop pair, rank the two sides by unburned_pixels (descending).
# Ranking happens WITHIN each drop_id group — each drop's sides are only
# compared against each other, not against unrelated drops from other fires.
df_rule = df_model_clean.copy()
df_rule["rank_unburned"] = df_rule.groupby("drop_id")["unburned_pixels"].rank(ascending=False)

# rank=1 (more unburned) → predicted protected (1)
# rank=2 (less unburned) → predicted origin (0)
df_rule["rule_pred"] = (df_rule["rank_unburned"] == 1).astype(int)

rule_acc = accuracy_score(df_rule["pt"], df_rule["rule_pred"])
print(f"Rule baseline accuracy: {rule_acc:.3f}")
print(f"Evaluated on {len(df_rule)} rows ({df_rule['drop_id'].nunique()} clean drop pairs)")
# Result: ~69% — this is the baseline the ML model must beat

Rule baseline accuracy: 0.694
Evaluated on 20472 rows (10236 clean drop pairs)


# Approaches 2-4 - Random Forest Model Comparisons

Testing four feature configurations to understand what drives accuracy: 

**- Solo unburned_pixels** - can RF match the rule using the same single feature? 

**- Raw features only** - absolute pixel statistics without pairing context

**-Paired-difference only** - relative comparison between sides only

**-Raw + paired-difference** - best of both (our best DOB-only model)


In [ ]:
# Test 1: Solo unburned_pixels

run_model_cv(["unburned_pixels"], "Solo unburned_pixels (sanity check)")

# Test 2: Raw features only (original baseline)

run_model_cv(raw_features, "Raw features only")

# Test 3: Paired-difference features only
# Directly encodes 'how does this side compare to its sibling'.

run_model_cv(diff_features, "Paired-difference features only")

# Test 4: Raw + paired-difference combined (BEST DOB MODEL)
# Combines absolute pixel statistics with relative paired comparisons.
acc_dob, std_dob, model_dob = run_model_cv(dob_features, "Raw + paired-difference combined (DOB best)")


Solo unburned_pixels (sanity check)
Accuracy: 0.578 ± 0.008
Per-fold: [0.567, 0.582, 0.587, 0.584, 0.571]
Top features: {'unburned_pixels': np.float64(1.0)}
--------------------------------------------------

Raw features only
Accuracy: 0.695 ± 0.006
Per-fold: [0.686, 0.701, 0.701, 0.698, 0.691]
Top features: {'burned_pixels': np.float64(0.26424367458658626), 'unburned_pixels': np.float64(0.1753004103409461), 'mean_dob': np.float64(0.16593856150724143), 'median_dob': np.float64(0.1393062673666433), 'total_pixels': np.float64(0.12936166133435437)}
--------------------------------------------------

Paired-difference features only
Accuracy: 0.800 ± 0.012
Per-fold: [0.796, 0.808, 0.787, 0.819, 0.791]
Top features: {'burned_pair_diff': np.float64(0.555039655549691), 'unburned_pair_diff': np.float64(0.2929563189206693), 'early_pair_diff': np.float64(0.1520040255296396)}
--------------------------------------------------

Raw + paired-difference combined (DOB best)
Accuracy: 0.813 ± 0.008
P

# Approach 5 - DOB + Fuel Features Combined

In [9]:
# Test fuel features alone — does fuel type carry independent signal?
run_model_cv(fuel_features, "Fuel features only")

# Test DOB + fuel combined — does adding fuel improve over DOB alone?
acc_fuel, std_fuel, model_fuel = run_model_cv(dob_features + fuel_features, "DOB + fuel combined")


Fuel features only
Accuracy: 0.552 ± 0.015
Per-fold: [0.533, 0.544, 0.555, 0.579, 0.551]
Top features: {'shrub_pixels': np.float64(0.15730118579145608), 'shrub_pair_diff': np.float64(0.14246696465327008), 'timber_pixels': np.float64(0.1325470677103274), 'timber_pair_diff': np.float64(0.1262477961563368), 'grass_pair_diff': np.float64(0.12252433033119418)}
--------------------------------------------------

DOB + fuel combined
Accuracy: 0.813 ± 0.011
Per-fold: [0.807, 0.806, 0.813, 0.834, 0.806]
Top features: {'burned_pair_diff': np.float64(0.29243421435946393), 'unburned_pair_diff': np.float64(0.11504291273535783), 'burned_pixels': np.float64(0.08693938671764473), 'early_pair_diff': np.float64(0.057738039405815476), 'mean_dob': np.float64(0.04820880138802654)}
--------------------------------------------------


# Approach 6 -- Whole-Raster Split (Best Model)

**Motivation**: DOB and fuel features sampled only the small buffered polygon, which plateaued around 81-82% regardless of feature combination or algorithm choice. The raster split approach instead extends the drop line across the *entire* fire raster, splitting it into two halves and comparing whole-fire burn statistics -- thousands of pixels per side instead of few hundred. 

**Validated visually in QGIS** on multiple random drops before implementation. 

See Notebook 2, Part 3 for the extraction methodology (`extract_split_features`). 

In [10]:
# Test split features alone — whole-fire comparison, no local buffer info
acc_split, std_split, model_split = run_model_cv(split_features, "Split features only")

# Test DOB + split combined — local buffer + whole-fire context together
acc_dob_split, std_dob_split, model_dob_split = run_model_cv(
    dob_features + split_features, "DOB + split combined (BEST)"
)

# Test everything combined — DOB + fuel + split
acc_all, std_all, model_all = run_model_cv(all_features, "DOB + fuel + split (all features)")


Split features only
Accuracy: 0.841 ± 0.023
Per-fold: [0.831, 0.828, 0.823, 0.839, 0.886]
Top features: {'split_burned_pair_diff': np.float64(0.27970183983053676), 'split_early_pair_diff': np.float64(0.1923397960219618), 'split_unburned_pair_diff': np.float64(0.13375183163329735), 'split_burned_pixels': np.float64(0.10180456549493803), 'split_early_pixels': np.float64(0.06502304105820371)}
--------------------------------------------------

DOB + split combined (BEST)
Accuracy: 0.901 ± 0.014
Per-fold: [0.901, 0.882, 0.89, 0.91, 0.923]
Top features: {'burned_pair_diff': np.float64(0.20697151161488062), 'split_burned_pair_diff': np.float64(0.16360204505875967), 'split_early_pair_diff': np.float64(0.1337604664551967), 'split_unburned_pair_diff': np.float64(0.08806951911269892), 'unburned_pair_diff': np.float64(0.05650060327636688)}
--------------------------------------------------

DOB + fuel + split (all features)
Accuracy: 0.901 ± 0.016
Per-fold: [0.9, 0.881, 0.886, 0.914, 0.923]
Top 

In [11]:

# Test everything combined — DOB + fuel + split + new_features # error fixing
acc_all, std_all, model_all = run_model_cv(new_features, "DOB + fuel + split (all features)")


DOB + fuel + split (all features)
Accuracy: 0.902 ± 0.015
Per-fold: [0.901, 0.882, 0.89, 0.914, 0.922]
Top features: {'burned_pair_diff': np.float64(0.15268210893888737), 'split_burned_pair_diff': np.float64(0.1473546365730426), 'split_early_pair_diff': np.float64(0.12372225505193668), 'split_unburned_pair_diff': np.float64(0.06590359785560403), 'split_burned_pixels': np.float64(0.054164494477238166)}
--------------------------------------------------


# Approach 7 -- XGBoost Confirmation

Testing whether a different algorithm can push past the Random Forest result.
If XGBoost matches Random Forest closely, that's strong evidence the ~90% ceiling
is a **data limitation**, not a modeling limitation — i.e. more powerful algorithms
won't help further without genuinely new information (e.g. wind data).

In [12]:
import xgboost as xgb

def run_xgb_cv(feature_list, label):
    X = df_model_clean[feature_list]
    accs = []
    importances = []

    for fold, (train_idx, test_idx) in enumerate(gkf.split(df_model_clean, y, groups)):
        model = xgb.XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            eval_metric="logloss",
            verbosity=0
        )
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = model.predict(X.iloc[test_idx])
        accs.append(accuracy_score(y.iloc[test_idx], preds))
        importances.append(model.feature_importances_)

    mean_acc = np.mean(accs)
    std_acc  = np.std(accs)
    mean_imp = np.mean(importances, axis=0)

    print(f"\n{label}")
    print(f"Accuracy: {mean_acc:.3f} ± {std_acc:.3f}")
    print(f"Per-fold: {[round(a, 3) for a in accs]}")
    print("Top features:", dict(sorted(zip(feature_list, mean_imp), key=lambda x: x[1], reverse=True)[:5]))
    print("-" * 50)
    return mean_acc, std_acc

acc_xgb, std_xgb = run_xgb_cv(dob_features + split_features, "XGBoost — DOB + split combined")

# Result: XGBoost ≈ Random Forest (~90.0%) → confirms 90% is a data ceiling,
# not a model choice limitation. Random Forest is kept as the final model
# (equally accurate, faster to train, simpler to explain).


XGBoost — DOB + split combined
Accuracy: 0.900 ± 0.016
Per-fold: [0.896, 0.882, 0.887, 0.909, 0.926]
Top features: {'split_burned_pair_diff': np.float32(0.37139067), 'burned_pair_diff': np.float32(0.21162629), 'split_early_pair_diff': np.float32(0.08033311), 'early_pair_diff': np.float32(0.03788068), 'burned_pixels': np.float32(0.03566835)}
--------------------------------------------------


---
## Save Final Model

Final model: Random Forest trained on DOB + split features (best accuracy/complexity
tradeoff — fuel added negligible improvement within noise, XGBoost matched but didn't
exceed Random Forest).

In [13]:
import joblib

final_features = dob_features + split_features

# Train final model on ALL available training data (not just one fold)
model_final = RandomForestClassifier(n_estimators=200, random_state=42)
model_final.fit(df_model_clean[final_features], y)

joblib.dump(model_final, "model_final.pkl")
joblib.dump(final_features, "model_features.pkl")
print(f"Model saved. Features used ({len(final_features)}): {final_features}")

Model saved. Features used (19): ['mean_dob', 'median_dob', 'early_pixels', 'late_pixels', 'unburned_pixels', 'burned_pixels', 'total_pixels', 'unburned_pair_diff', 'early_pair_diff', 'burned_pair_diff', 'split_mean_dob', 'split_early_pixels', 'split_late_pixels', 'split_unburned_pixels', 'split_burned_pixels', 'split_total_pixels', 'split_burned_pair_diff', 'split_unburned_pair_diff', 'split_early_pair_diff']


In [15]:
# ---- Error Analysis (Fixed) ----
train_idx, test_idx = list(gkf.split(df_model_clean, y, groups))[-1]

# train a FRESH model on train_idx only — never seen test_idx
model_eval = RandomForestClassifier(n_estimators=200, random_state=42)
model_eval.fit(df_model_clean[final_features].iloc[train_idx], y.iloc[train_idx])

# now predict on the truly unseen test_idx
X_test = df_model_clean[final_features].iloc[test_idx]
y_test = y.iloc[test_idx]
preds = model_eval.predict(X_test)

df_test = df_model_clean.iloc[test_idx].copy()
df_test["predicted"] = preds
df_test["correct"] = (df_test["predicted"] == df_test["pt"])

wrong = df_test[~df_test["correct"]]
print(f"Wrong predictions: {len(wrong)} / {len(df_test)}")

print("\n--- Feature stats for WRONG predictions ---")
print(wrong[["mean_dob", "split_burned_pixels", "split_unburned_pixels"]].describe())

print("\n--- Feature stats for CORRECT predictions ---")
print(df_test[df_test["correct"]][["mean_dob", "split_burned_pixels", "split_unburned_pixels"]].describe())

print("\n--- Fires with most errors ---")
print(wrong["event_id"].value_counts().head(20))

Wrong predictions: 317 / 4094

--- Feature stats for WRONG predictions ---
         mean_dob  split_burned_pixels  split_unburned_pixels
count  317.000000           317.000000           3.170000e+02
mean   201.561734         63820.984227           1.163039e+05
std     56.466333        110663.493326           1.783420e+05
min      0.000000            88.000000           1.159000e+03
25%    187.000000          9164.000000           1.563600e+04
50%    206.885246         27043.000000           4.830100e+04
75%    229.000000         73273.000000           1.129410e+05
max    297.000000        933842.000000           1.157147e+06

--- Feature stats for CORRECT predictions ---
          mean_dob  split_burned_pixels  split_unburned_pixels
count  3777.000000          3777.000000           3.777000e+03
mean    170.654764         70482.683347           1.248095e+05
std      90.925995        138683.731639           2.074845e+05
min       0.000000             0.000000           1.240000e+02
25%  

In [ ]:
# ---- Get Exact Failed Drops ----
wrong_drops = wrong[[
    "event_id", "drop_id", "buff_id", "pt", "predicted",
    "mean_dob", "split_burned_pixels", "split_unburned_pixels",
    "unburned_pixels", "burned_pixels"
]].copy()

print(f"Total failed predictions: {len(wrong_drops)}")
print(wrong_drops.head(20))

wrong_drops.to_csv("failed_predictions.csv", index=False)

Total failed predictions: 317
                   event_id  \
426   AZ3237411063220170630   
427   AZ3237411063220170630   
438   AZ3237411063220170630   
439   AZ3237411063220170630   
440   AZ3237411063220170630   
441   AZ3237411063220170630   
458   AZ3237411063220170630   
459   AZ3237411063220170630   
466   AZ3237411063220170630   
467   AZ3237411063220170630   
476   AZ3237411063220170630   
507   AZ3237411063220170630   
508   AZ3237411063220170630   
509   AZ3237411063220170630   
510   AZ3237411063220170630   
516   AZ3237411063220170630   
519   AZ3237411063220170630   
520   AZ3237411063220170630   
1157  AZ3395311201220190903   
1158  AZ3395311201220190903   

                                                drop_id  buff_id  pt  \
426   AZ3237411063220170630_32.3883_-110.67127_2017-...      1.0   0   
427   AZ3237411063220170630_32.3883_-110.67127_2017-...      2.0   1   
438   AZ3237411063220170630_32.38688_-110.67029_2017...      1.0   1   
439   AZ3237411063220170630_32

In [26]:
import geopandas as gpd

# ---- Load Clean Drops ----
drops = gpd.read_file("drops_clean.geojson")
print(f"Loaded {len(drops)} drop sides across {drops['Evnt_ID'].nunique()} fires")

Loaded 21234 drop sides across 424 fires


In [27]:
# merge back with the original drops geodataframe to get geometry
failed_geoms = drops[drops["drop_id"].isin(wrong_drops["drop_id"])]
failed_geoms.to_file("failed_predictions.geojson", driver="GeoJSON")
print(f"Saved {len(failed_geoms)} failed drop geometries for QGIS inspection")

Saved 358 failed drop geometries for QGIS inspection


In [29]:
print(wrong_drops[wrong_drops["drop_id"].str.contains("WY4304211058720180915_43.10596_-110.22533_2018-09-23 23:08:49 UTC")]["drop_id"].unique())

['WY4304211058720180915_43.10596_-110.22533_2018-09-23 23:08:49 UTC']


In [31]:
# check how often local and split features disagree in direction
wrong["local_says_protected"] = wrong["unburned_pixels"] > wrong["burned_pixels"]
wrong["split_says_protected"] = wrong["split_unburned_pixels"] > wrong["split_burned_pixels"]
wrong["features_disagree"] = wrong["local_says_protected"] != wrong["split_says_protected"]

print(f"Disagreement rate in WRONG predictions: {wrong['features_disagree'].mean():.1%}")

Disagreement rate in WRONG predictions: 24.9%


/var/folders/gp/hgxpjqv54rg2r22rtswcl76r0000gn/T/ipykernel_40321/2767055386.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wrong["local_says_protected"] = wrong["unburned_pixels"] > wrong["burned_pixels"]
/var/folders/gp/hgxpjqv54rg2r22rtswcl76r0000gn/T/ipykernel_40321/2767055386.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wrong["split_says_protected"] = wrong["split_unburned_pixels"] > wrong["split_burned_pixels"]
/var/folders/gp/hgxpjqv54rg2r22rtswcl76r0000gn/T/ipykernel_40321/2767055386.py:4

---
# Approach 8 -- Post-Meeting Feature Engineering (Pushing Toward 95%)

After reviewing the 90.1% result and the error analysis above, the professor's guidance was to push
accuracy **as high as possible** (target ~95-96%), rather than treat 90% as a stopping point. Two
failure modes were identified from the error analysis:

1. Local-buffer vs. raster-split feature disagreement (~25% of wrong predictions)
2. Complex/irregular fire shapes breaking the whole-raster split



## 8a -- + Pre-Drop Burned Pixels

In [17]:
# ---- Load All Feature CSVs ----
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df_dob     = pd.read_csv("training_features_dob.csv")
df_split   = pd.read_csv("training_features_split.csv")
df_dist    = pd.read_csv("training_features_dist.csv")
df_predrop = pd.read_csv("training_features_predrop.csv")

# ---- Merge All ----
df = df_dob.merge(df_split,   on=["event_id","buff_id","drop_id"], how="left")
df = df.merge(df_dist,        on=["event_id","buff_id","drop_id"], how="left")
df = df.merge(df_predrop,     on=["event_id","buff_id","drop_id"], how="left")

print(f"Total rows: {len(df)}")

# ---- Filter Clean Pairs ----
group_sizes = df.groupby("drop_id").size()
clean_drop_ids = group_sizes[group_sizes == 2].index
df_model = df[df["drop_id"].isin(clean_drop_ids)].copy()
print(f"Clean pairs: {len(df_model)} rows")

# ---- Fix NaNs ----
df_model["mean_dob"]              = df_model["mean_dob"].fillna(0)
df_model["median_dob"]            = df_model["median_dob"].fillna(0)
df_model["split_mean_dob"]        = df_model["split_mean_dob"].fillna(0)
df_model["predrop_mean_dob"]      = df_model["predrop_mean_dob"].fillna(0)
df_model["predrop_burned_pixels"] = df_model["predrop_burned_pixels"].fillna(0)

# ---- Build All Paired-Difference Features ----
# DOB
for col in ["unburned_pixels", "early_pixels", "burned_pixels"]:
    df_model[f"{col.replace('_pixels','')}_pair_diff"] = \
        df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())

# Split
for col in ["split_unburned_pixels", "split_early_pixels", "split_burned_pixels"]:
    new_col = col.replace("_pixels","") + "_pair_diff"
    df_model[new_col] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())

# Distance
df_model["dist_pair_diff"] = df_model.groupby("drop_id")["dist_to_unburned"].transform(
    lambda x: x - x.mean()
)

# Pre-drop
df_model["predrop_pair_diff"] = df_model.groupby("drop_id")["predrop_burned_pixels"].transform(
    lambda x: x - x.mean()
)

# ---- Define Feature Sets ----
dob_features = [
    "mean_dob", "median_dob", "early_pixels", "late_pixels",
    "unburned_pixels", "burned_pixels", "total_pixels",
    "unburned_pair_diff", "early_pair_diff", "burned_pair_diff"
]

split_features = [
    "split_mean_dob", "split_early_pixels", "split_late_pixels",
    "split_unburned_pixels", "split_burned_pixels", "split_total_pixels",
    "split_burned_pair_diff", "split_unburned_pair_diff", "split_early_pair_diff"
]

dist_features = [
    "dist_to_unburned", "dist_pair_diff"
]

predrop_features = [
    "predrop_burned_pixels", "predrop_mean_dob",
    "predrop_total_pixels", "predrop_pair_diff"
]

all_features = dob_features + split_features + dist_features + predrop_features

# ---- Drop NaN Rows ----
df_model_clean = df_model.dropna(subset=all_features + ["pt"])
print(f"Rows for training: {len(df_model_clean)}")

# ---- y, groups, gkf ----
y      = df_model_clean["pt"]
groups = df_model_clean["event_id"]
gkf    = GroupKFold(n_splits=5)

# ---- Model Function ----
def run_model_cv(feature_list, label):
    X = df_model_clean[feature_list]
    accs = []
    importances = []

    for fold, (train_idx, test_idx) in enumerate(gkf.split(df_model_clean, y, groups)):
        model = RandomForestClassifier(n_estimators=200, random_state=42)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = model.predict(X.iloc[test_idx])
        accs.append(accuracy_score(y.iloc[test_idx], preds))
        importances.append(model.feature_importances_)

    mean_acc = np.mean(accs)
    std_acc  = np.std(accs)
    mean_imp = np.mean(importances, axis=0)

    print(f"\n{label}")
    print(f"Accuracy: {mean_acc:.3f} ± {std_acc:.3f}")
    print(f"Per-fold: {[round(a, 3) for a in accs]}")
    top5 = dict(sorted(zip(feature_list, mean_imp),
                key=lambda x: x[1], reverse=True)[:5])
    print(f"Top features: {top5}")
    print("-" * 50)
    return mean_acc, std_acc

# ---- Run Comparisons ----
run_model_cv(dob_features + split_features,         "DOB + split (baseline)")
run_model_cv(dob_features + split_features + predrop_features, "DOB + split + predrop")
run_model_cv(all_features,                           "All features combined")

Total rows: 21081
Clean pairs: 20478 rows
Rows for training: 20472

DOB + split (baseline)
Accuracy: 0.901 ± 0.014
Per-fold: [0.901, 0.882, 0.89, 0.91, 0.923]
Top features: {'burned_pair_diff': np.float64(0.20697151161488062), 'split_burned_pair_diff': np.float64(0.16360204505875967), 'split_early_pair_diff': np.float64(0.1337604664551967), 'split_unburned_pair_diff': np.float64(0.08806951911269892), 'unburned_pair_diff': np.float64(0.05650060327636688)}
--------------------------------------------------

DOB + split + predrop
Accuracy: 0.901 ± 0.014
Per-fold: [0.9, 0.884, 0.889, 0.909, 0.923]
Top features: {'burned_pair_diff': np.float64(0.180214034360632), 'split_burned_pair_diff': np.float64(0.16401733196905388), 'predrop_pair_diff': np.float64(0.09367291946479112), 'split_early_pair_diff': np.float64(0.08969512828662016), 'split_unburned_pair_diff': np.float64(0.05294079496751695)}
--------------------------------------------------

All features combined
Accuracy: 0.908 ± 0.015
Per

(np.float64(0.9078252854787493), np.float64(0.014668509626770755))

## 8b -- + Distance to Nearest Unburned + Signal-Agreement Features

`signal_agreement`, `local_signal_strength`, `split_signal_strength`, and `split_dominance` were added specifically to try to resolve the local-vs-split disagreement cases identified in the error analysis above.

In [22]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

# load all feature CSVs
df_dob   = pd.read_csv("training_features_dob.csv")
df_split = pd.read_csv("training_features_split.csv")
df_dist  = pd.read_csv("training_features_dist.csv")
df_fuel  = pd.read_csv("training_features_fuel.csv") 


# merge everything together
df = df_dob.merge(df_split, on=["event_id","buff_id","drop_id"], how="left")
df = df.merge(df_dist,  on=["event_id","buff_id","drop_id"], how="left")
df = df.merge(df_fuel,      on=["event_id","buff_id","drop_id"], how="left")  


print(f"Total rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

Total rows: 21547
Columns: ['mean_dob', 'median_dob', 'early_pixels', 'late_pixels', 'unburned_pixels', 'total_pixels', 'burned_pixels', 'event_id', 'buff_id', 'altitude', 'knots', 'gallons', 'drop_seconds', 'pt', 'drop_id', 'split_mean_dob', 'split_early_pixels', 'split_late_pixels', 'split_unburned_pixels', 'split_burned_pixels', 'split_total_pixels', 'dist_to_unburned', 'dominant_fuel', 'fuel_diversity', 'grass_pixels', 'shrub_pixels', 'timber_pixels', 'slash_pixels', 'nonburnable_pixels', 'total_fuel_pixels']


In [28]:
# Preparing a Base DataFrame
# Filter to clean drop pairs (exactly 2 rows per drop_id)
# Drops with 1, 3, 4, or 6 rows (~2.4%) are excluded — ambiguous pairs
group_sizes = df.groupby("drop_id").size()
clean_drop_ids = group_sizes[group_sizes == 2].index
df_model = df[df["drop_id"].isin(clean_drop_ids)].copy()
print(f"Clean pairs: {len(df_model)} rows ({df_model['drop_id'].nunique()} drops)")

# Fix NaN mean_dob/median_dob/split_mean_dob — polygons where nothing burned.
# These rows are valid training examples (burned_pixels=0 already captures this).
# Filling with 0 keeps them rather than silently dropping rows.
df_model["mean_dob"]       = df_model["mean_dob"].fillna(0)
df_model["median_dob"]     = df_model["median_dob"].fillna(0)
df_model["split_mean_dob"] = df_model["split_mean_dob"].fillna(0)

# Building Paired-Difference Feature
# For each drop pair, compute how much each side differs from the pair's average.
# Positive diff = more than sibling, negative diff = less than sibling.
# This is the key insight: relative comparison between sides matters more than
# absolute values, because it's self-normalizing across drops of very different sizes.

# DOB paired-difference features (local buffer)
for col in ["unburned_pixels", "early_pixels", "burned_pixels"]:
    df_model[f"{col.replace('_pixels', '')}_pair_diff"] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())

# Fuel paired-difference features
for col in ["grass_pixels", "shrub_pixels", "timber_pixels", "nonburnable_pixels"]:
    df_model[f"{col.replace('_pixels', '')}_pair_diff"] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())

# Split paired-difference features (whole-raster comparison — strongest signal)
for col in ["split_unburned_pixels", "split_early_pixels", "split_burned_pixels"]:
    new_col = col.replace("split_", "split_").replace("_pixels", "") + "_pair_diff"
    df_model[new_col] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())


df_model["dist_pair_diff"] = df_model.groupby("drop_id")["dist_to_unburned"].transform(
    lambda x: x - x.mean()
)

# New feature addition trying to fix the error. 25% of the ~10% error was due to disagreement between local pixels and split pixels information. we are trying to see the disagreement and train accordingly. 

df_model["signal_agreement"] = (
    df_model["burned_pair_diff"] * df_model["split_burned_pair_diff"]
)
df_model["local_signal_strength"] = df_model["burned_pair_diff"].abs()
df_model["split_signal_strength"] = df_model["split_burned_pair_diff"].abs()
df_model["split_dominance"] = (
    df_model["split_signal_strength"] /
    (df_model["local_signal_strength"] + df_model["split_signal_strength"] + 1e-9)
)

print(f"DataFrame shape after feature engineering: {df_model.shape}")

Clean pairs: 20474 rows (10237 drops)
DataFrame shape after feature engineering: (20474, 45)


In [31]:
# Defining Feature Sets

raw_features = [
    "mean_dob", "median_dob", "early_pixels", "late_pixels",
    "unburned_pixels", "burned_pixels", "total_pixels"
]

diff_features = [
    "unburned_pair_diff", "early_pair_diff", "burned_pair_diff"
]

dob_features = raw_features + diff_features

fuel_features = [
    "dominant_fuel", "fuel_diversity",
    "grass_pixels", "shrub_pixels", "timber_pixels",
    "slash_pixels", "nonburnable_pixels",
    "grass_pair_diff", "shrub_pair_diff",
    "timber_pair_diff", "nonburnable_pair_diff"
]

# Whole-raster split features — the strongest predictor discovered.
# Same statistics as DOB features, but computed across the ENTIRE fire raster
# split by the extended drop line, rather than just the small buffer polygon.
split_features = [
    "split_mean_dob", "split_early_pixels", "split_late_pixels",
    "split_unburned_pixels", "split_burned_pixels", "split_total_pixels",
    "split_burned_pair_diff", "split_unburned_pair_diff", "split_early_pair_diff"
]



all_features = dob_features + fuel_features + split_features


# New fix after error analysis

agreement_features = [
    "signal_agreement",
    "local_signal_strength",
    "split_signal_strength",
    "split_dominance"
]

new_features = dob_features + split_features + [
    "dist_to_unburned", 
    "dist_pair_diff"
]


In [30]:
# ---- Final Clean DataFrame ----
# Drop rows with NaN in any feature we'll use.
# Note: dominant_fuel NaN = polygon outside fuel data coverage (very rare)
df_model_clean = df_model.dropna(subset=new_features + ["pt"])
print(f"Rows for training: {len(df_model_clean)}")
print(f"Unique fires: {df_model_clean['event_id'].nunique()}")

# ---- Define y, groups, gkf ONCE ----
# These are defined here and never redefined below.
# All model comparisons use the same split for fair evaluation.
y      = df_model_clean["pt"]      # target: 1=protected, 0=origin
groups = df_model_clean["event_id"] # split by FIRE to prevent leakage
gkf    = GroupKFold(n_splits=5)

Rows for training: 20472
Unique fires: 420


In [32]:


run_model_cv(new_features, "DOB + split + distance")


DOB + split + distance
Accuracy: 0.908 ± 0.014
Per-fold: [0.911, 0.888, 0.897, 0.918, 0.924]
Top features: {'split_burned_pair_diff': np.float64(0.1547658587208958), 'burned_pair_diff': np.float64(0.15335423090235362), 'dist_pair_diff': np.float64(0.15124193458662422), 'split_early_pair_diff': np.float64(0.10975688038985323), 'split_unburned_pair_diff': np.float64(0.06217047159437024)}
--------------------------------------------------


(np.float64(0.9076295913226673), np.float64(0.013505550214302872))

## 8c -- + Expanded Polygon (Final Push)

The expanded-polygon features (Notebook 2, Part 4c) turned out to be the strongest addition -- `exp_burned_pair_diff` becomes the top-ranked feature in the combined model.

In [ ]:
df_exp = pd.read_csv("training_features_expanded.csv")

# merge with existing
df_all = pd.read_csv("training_features_dob.csv")
df_all = df_all.merge(pd.read_csv("training_features_split.csv"), on=["event_id","buff_id","drop_id"], how="left")
df_all = df_all.merge(pd.read_csv("training_features_dist.csv"),  on=["event_id","buff_id","drop_id"], how="left")
df_all = df_all.merge(df_exp, on=["event_id","buff_id","drop_id"], how="left")

# filter clean pairs
group_sizes = df_all.groupby("drop_id").size()
clean_drop_ids = group_sizes[group_sizes == 2].index
df_model = df_all[df_all["drop_id"].isin(clean_drop_ids)].copy()

# fix NaNs
for col in ["mean_dob","median_dob","split_mean_dob","exp_mean_dob"]:
    df_model[col] = df_model[col].fillna(0)

# DOB paired-diff
for col in ["unburned_pixels","early_pixels","burned_pixels"]:
    df_model[f"{col.replace('_pixels','')}_pair_diff"] = \
        df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())

# split paired-diff
for col in ["split_unburned_pixels","split_early_pixels","split_burned_pixels"]:
    new_col = col.replace("_pixels","") + "_pair_diff"
    df_model[new_col] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())

# distance paired-diff
df_model["dist_pair_diff"] = df_model.groupby("drop_id")["dist_to_unburned"].transform(
    lambda x: x - x.mean()
)

# expanded paired-diff
for col in ["exp_burned_pixels","exp_unburned_pixels","exp_early_pixels"]:
    new_col = col.replace("_pixels","") + "_pair_diff"
    df_model[new_col] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())

# feature sets
dob_features = [
    "mean_dob","median_dob","early_pixels","late_pixels",
    "unburned_pixels","burned_pixels","total_pixels",
    "unburned_pair_diff","early_pair_diff","burned_pair_diff"
]
split_features = [
    "split_mean_dob","split_early_pixels","split_late_pixels",
    "split_unburned_pixels","split_burned_pixels","split_total_pixels",
    "split_burned_pair_diff","split_unburned_pair_diff","split_early_pair_diff"
]
dist_features = ["dist_to_unburned","dist_pair_diff"]
exp_features = [
    "exp_mean_dob","exp_burned_pixels","exp_unburned_pixels",
    "exp_early_pixels","exp_late_pixels","exp_total_pixels",
    "exp_burned_pair_diff","exp_unburned_pair_diff","exp_early_pair_diff"
]

all_features = dob_features + split_features + dist_features + exp_features

df_model_clean = df_model.dropna(subset=all_features + ["pt"])
print(f"Rows: {len(df_model_clean)}")

y      = df_model_clean["pt"]
groups = df_model_clean["event_id"]
gkf    = GroupKFold(n_splits=5)

def run_model_cv(feature_list, label):
    X = df_model_clean[feature_list]
    accs = []
    importances = []
    for fold, (train_idx, test_idx) in enumerate(gkf.split(df_model_clean, y, groups)):
        model = RandomForestClassifier(n_estimators=200, random_state=42)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = model.predict(X.iloc[test_idx])
        accs.append(accuracy_score(y.iloc[test_idx], preds))
        importances.append(model.feature_importances_)
    mean_acc = np.mean(accs)
    std_acc  = np.std(accs)
    mean_imp = np.mean(importances, axis=0)
    print(f"\n{label}")
    print(f"Accuracy: {mean_acc:.3f} ± {std_acc:.3f}")
    print(f"Per-fold: {[round(a,3) for a in accs]}")
    top5 = dict(sorted(zip(feature_list, mean_imp), key=lambda x: x[1], reverse=True)[:5])
    print(f"Top features: {top5}")
    print("-"*50)
    return mean_acc, std_acc

# run comparisons
run_model_cv(dob_features + split_features,          "DOB + split (baseline)")
run_model_cv(exp_features,                            "Expanded polygon only")
run_model_cv(dob_features + split_features + exp_features, "DOB + split + expanded")
run_model_cv(all_features,                            "All features combined")

Rows: 20472

DOB + split (baseline)
Accuracy: 0.901 ± 0.014
Per-fold: [0.901, 0.882, 0.89, 0.91, 0.923]
Top features: {'burned_pair_diff': np.float64(0.20697151161488062), 'split_burned_pair_diff': np.float64(0.16360204505875967), 'split_early_pair_diff': np.float64(0.1337604664551967), 'split_unburned_pair_diff': np.float64(0.08806951911269892), 'unburned_pair_diff': np.float64(0.05650060327636688)}
--------------------------------------------------

Expanded polygon only
Accuracy: 0.879 ± 0.020
Per-fold: [0.866, 0.873, 0.853, 0.891, 0.911]
Top features: {'exp_burned_pair_diff': np.float64(0.3830510766809206), 'exp_early_pair_diff': np.float64(0.2067387121423423), 'exp_burned_pixels': np.float64(0.10810011667377613), 'exp_early_pixels': np.float64(0.062025247949728336), 'exp_unburned_pixels': np.float64(0.05019425043384148)}
--------------------------------------------------

DOB + split + expanded
Accuracy: 0.913 ± 0.013
Per-fold: [0.905, 0.903, 0.901, 0.927, 0.929]
Top features: {'e

(np.float64(0.9171068512991575), np.float64(0.012378391094704037))

## 8d -- XGBoost Confirmation on Full Feature Set

Same check as Approach 7: if a different algorithm matches Random Forest, the ceiling is a data limitation, not a model-choice limitation.

In [34]:
import xgboost as xgb

def run_xgb_cv(feature_list, label):
    X = df_model_clean[feature_list]
    accs = []
    for fold, (train_idx, test_idx) in enumerate(gkf.split(df_model_clean, y, groups)):
        model = xgb.XGBClassifier(n_estimators=200, max_depth=6,
                                   learning_rate=0.1, random_state=42,
                                   eval_metric="logloss", verbosity=0)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = model.predict(X.iloc[test_idx])
        accs.append(accuracy_score(y.iloc[test_idx], preds))
    print(f"{label}: {np.mean(accs):.3f} ± {np.std(accs):.3f}")

run_xgb_cv(all_features, "XGBoost — all features")

XGBoost — all features: 0.919 ± 0.012


## Final Model (v2) -- 91.9%

Random Forest and XGBoost converge to the same accuracy on the full feature set (DOB + split + distance + pre-drop + expanded polygon). XGBoost is saved as the final model since it gave the marginally higher result.

In [35]:
import joblib
final_features = all_features
model_v2 = xgb.XGBClassifier(n_estimators=200, max_depth=6,
                               learning_rate=0.1, random_state=42,
                               eval_metric="logloss", verbosity=0)
model_v2.fit(df_model_clean[final_features], y)
joblib.dump(model_v2, "model_final_v2.pkl")
joblib.dump(final_features, "model_features_v2.pkl")
print("Saved ✅")

Saved ✅
